# Chapter 4: The Governed Tool Set and Policy Store

**Learning objectives:**
1. Design tool schemas in JSON format (compatible with Claude tool_use and standalone use)
2. Implement sklearn-based tool handlers returning both display strings and raw Python dicts
3. Build SessionState for cross-tool state sharing
4. Test each tool independently

In [ ]:
# === Preamble (all notebooks) ===

import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}, CUDA: {torch.cuda.is_available()}")


## Key Concepts

| Term | Definition |
|------|------------|
| **Dual-return pattern** | Every tool handler returns `(display_string, raw_dict)`. The display string is human-readable; the raw dict is pure Python and passes `json.dumps` without error, enabling Tier 1 deterministic extraction. |
| **SessionState** | A mutable dataclass (datasets, models, results, preprocessors, predictions) passed by reference so that any tool's output is visible to all subsequent tools in the same session. |
| **JSON serializability** | The `_to_python` helper recursively converts NumPy/pandas types to native Python, guaranteeing every raw dict is JSON-serializable. |
| **Tool groups** | 15 tools in 5 groups: Data (4), Model (4), Diagnostic (3), Explanation (2), Agent stubs (2). |
| **Permutation importance** | Measures accuracy drop when a single feature is randomly shuffled. Large drop = important feature. |
| **SHAP** | SHapley Additive exPlanations: assigns each feature an additive contribution to each prediction, grounded in cooperative game theory. |
| **Adverse Impact Ratio (AIR)** | Ratio of positive-outcome rates between groups. AIR < 0.80 (the 4/5 rule) indicates potential disparate impact. |
| **Robustness diagnosis** | Adds Gaussian noise at increasing levels and measures accuracy degradation, simulating data drift and measurement error. |

In [ ]:
# --- Inspect the tool library ---
from knowlytix.harness.governance.session import SessionState
from knowlytix.harness.governance.tools import TOOL_DEFINITIONS, execute_tool

print(f"SessionState fields: {list(SessionState.__dataclass_fields__.keys())}")
print(f"\nTool catalog ({len(TOOL_DEFINITIONS)} tools):")
print("-" * 55)
for t in TOOL_DEFINITIONS:
    n_params = len(t["input_schema"].get("properties", {}))
    required = t["input_schema"].get("required", [])
    print(f"  {t['name']:<22s}  params={n_params}  required={required}")

## Lab 4.1 --- Build and Test Data Tools

Load Iris, run `data_summary`, `eda_correlation` and `preprocess_data`. Show display strings and raw dicts. Verify state is populated after each call.

In [3]:
# --- Lab 4.1a: load_dataset ---
state = SessionState()

display, raw = execute_tool("load_dataset", {"name": "iris", "builtin": "iris"}, state)
print("DISPLAY:")
print(display)
print("\nRAW DICT:")
for k, v in raw.items():
    if k != "feature_names":
        print(f"  {k}: {v}")
    else:
        print(f"  {k}: {v[:3]}... ({len(v)} total)")

print(f"\nState check: 'iris' in state.datasets = {'iris' in state.datasets}")

DISPLAY:
Loaded 'iris': 150 samples, 4 features, target='target', task=classification
  Train: 120, Test: 30

RAW DICT:
  name: iris
  n_samples: 150
  n_features: 4
  target_name: target
  task_type: classification
  train_size: 120
  test_size: 30
  feature_names: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)']... (4 total)

State check: 'iris' in state.datasets = True


In [4]:
# --- Lab 4.1b: data_summary ---
display, raw = execute_tool("data_summary", {"dataset": "iris"}, state)
print("DISPLAY (first 400 chars):")
print(display[:400])
print("\nRAW DICT keys:", list(raw.keys()))
print(f"  n_rows: {raw['n_rows']}, n_cols: {raw['n_cols']}")
print(f"  missing%: {raw['missing_pct']}")

DISPLAY (first 400 chars):
Data summary for 'iris':
       sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)      target
count         150.000000        150.000000         150.000000        150.000000  150.000000
mean            5.843333          3.057333           3.758000          1.199333    1.000000
std             0.828066          0.435866           1.765298          0.762238    0.819232
min    

RAW DICT keys: ['describe', 'dtypes', 'missing_pct', 'n_rows', 'n_cols']
  n_rows: 150, n_cols: 5
  missing%: {'sepal length (cm)': 0.0, 'sepal width (cm)': 0.0, 'petal length (cm)': 0.0, 'petal width (cm)': 0.0, 'target': 0.0}


In [5]:
# --- Lab 4.1c: eda_correlation ---
display, raw = execute_tool("eda_correlation",
                            {"dataset": "iris", "method": "pearson"}, state)
print("DISPLAY (first 300 chars):")
print(display[:300])
print("\nRAW keys:", list(raw.keys()))
print(f"  method: {raw['method']}")
# Show top correlation pair (excluding self-correlations)
corr = raw["correlation"]
features = list(corr.keys())
top_pair, top_val = None, 0
for f1 in features:
    for f2 in features:
        if f1 < f2:
            val = abs(corr[f1][f2])
            if val > top_val:
                top_val = val
                top_pair = (f1, f2)
print(f"  Highest |corr|: {top_pair} = {top_val:.4f}")

DISPLAY (first 300 chars):
Correlation matrix (pearson):
                   sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)    target
sepal length (cm)           1.000000         -0.117570           0.871754          0.817941  0.782561
sepal width (cm)           -0.117570          1.000000          -0

RAW keys: ['method', 'correlation']
  method: pearson
  Highest |corr|: ('petal length (cm)', 'petal width (cm)') = 0.9629


In [6]:
# --- Lab 4.1d: preprocess_data ---
display, raw = execute_tool("preprocess_data",
                            {"dataset": "iris", "impute_method": "mean",
                             "scale_method": "standard", "encode_method": "ordinal"},
                            state)
print("DISPLAY:")
print(display)
print("\nRAW DICT:")
for k, v in raw.items():
    print(f"  {k}: {v}")

print(f"\nState check: 'iris' in state.preprocessors = {'iris' in state.preprocessors}")

DISPLAY:
Preprocessing done: impute=mean, scale=standard, encode=ordinal
  Features: 4 -> 4
  Numeric cols: 4, Categorical cols: 0

RAW DICT:
  impute_method: mean
  scale_method: standard
  encode_method: ordinal
  n_features_before: 4
  n_features_after: 4
  numeric_cols: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
  categorical_cols: []

State check: 'iris' in state.preprocessors = True


In [7]:
# --- Lab 4.1e: Verify return types are pure Python ---
import json

print("Verifying raw dicts are JSON-serializable (pure Python):")
for tool_name in ["load_dataset", "data_summary", "eda_correlation", "preprocess_data"]:
    if tool_name == "load_dataset":
        args = {"name": "iris_check", "builtin": "iris"}
    elif tool_name == "data_summary":
        args = {"dataset": "iris"}
    elif tool_name == "eda_correlation":
        args = {"dataset": "iris"}
    else:
        args = {"dataset": "iris"}
    _, raw = execute_tool(tool_name, args, state)
    try:
        json.dumps(raw)
        print(f"  {tool_name:<22s}  OK  (keys: {list(raw.keys())})")
    except TypeError as e:
        print(f"  {tool_name:<22s}  FAIL: {e}")

Verifying raw dicts are JSON-serializable (pure Python):
  load_dataset            OK  (keys: ['name', 'n_samples', 'n_features', 'target_name', 'task_type', 'train_size', 'test_size', 'feature_names'])
  data_summary            OK  (keys: ['describe', 'dtypes', 'missing_pct', 'n_rows', 'n_cols'])
  eda_correlation         OK  (keys: ['method', 'correlation'])
  preprocess_data         OK  (keys: ['impute_method', 'scale_method', 'encode_method', 'n_features_before', 'n_features_after', 'numeric_cols', 'categorical_cols'])


## Lab 4.2 --- Build and Test Model + Diagnostic Tools

Train three models on Iris (LogisticRegression, RandomForest, GradientBoosting). Run `diagnose_robustness`, `explain_importance`. Run `leaderboard` to compare.

In [8]:
# --- Lab 4.2a: Feature selection ---
display, raw = execute_tool("feature_select",
                            {"dataset": "iris", "method": "f_classif", "k": 4},
                            state)
print("DISPLAY:")
print(display)
print("\nRAW selected features:", raw["selected_features"])

DISPLAY:
Feature selection (f_classif, k=4):
  petal length (cm): 925.5564
  petal width (cm): 680.7756
  sepal length (cm): 84.8084
  sepal width (cm): 41.2928

RAW selected features: ['petal length (cm)', 'petal width (cm)', 'sepal length (cm)', 'sepal width (cm)']


In [9]:
# --- Lab 4.2b: Train three models ---
models_to_train = [
    ("lr",  "LogisticRegression",         {"max_iter": 200, "random_state": 42}),
    ("rf",  "RandomForestClassifier",     {"n_estimators": 100, "random_state": 42}),
    ("gb",  "GradientBoostingClassifier", {"n_estimators": 100, "random_state": 42}),
]

for mname, mtype, params in models_to_train:
    display, raw = execute_tool("train_model", {
        "dataset": "iris",
        "model_type": mtype,
        "model_name": mname,
        "params": params,
    }, state)
    print(display)
    print()

Trained 'lr' (LogisticRegression):
  train_accuracy: 0.975
  train_f1_macro: 0.975
  train_auc_ovr: 0.9981
  test_accuracy: 1.0
  test_f1_macro: 1.0
  test_auc_ovr: 1.0
  train_test_gap: -0.025

Trained 'rf' (RandomForestClassifier):
  train_accuracy: 1.0
  train_f1_macro: 1.0
  train_auc_ovr: 1.0
  test_accuracy: 1.0
  test_f1_macro: 1.0
  test_auc_ovr: 1.0
  train_test_gap: 0.0

Trained 'gb' (GradientBoostingClassifier):
  train_accuracy: 1.0
  train_f1_macro: 1.0
  train_auc_ovr: 1.0
  test_accuracy: 1.0
  test_f1_macro: 1.0
  test_auc_ovr: 1.0
  train_test_gap: 0.0



In [10]:
# --- Lab 4.2c: Predict ---
display, raw = execute_tool("predict",
                            {"dataset": "iris", "model_name": "rf"}, state)
print("DISPLAY:")
print(display)
print("\nRAW:", raw)
print(f"\nState check: 'rf' in state.predictions = {'rf' in state.predictions}")

DISPLAY:
Predictions for 'rf': 30 samples, unique values: 3

RAW: {'model_name': 'rf', 'n_predictions': 30, 'unique_values': [0, 1, 2]}

State check: 'rf' in state.predictions = True


In [11]:
# --- Lab 4.2d: Leaderboard ---
display, raw = execute_tool("leaderboard", {"dataset": "iris"}, state)
print("DISPLAY:")
print(display)
print("\nTop model:", raw["ranking"][0]["model"],
      "with", raw["sort_by"], "=", raw["ranking"][0][raw["sort_by"]])

DISPLAY:
Leaderboard:
Rank  Model                     Type                           test_accuracy  
------------------------------------------------------------------------------
1     lr                        LogisticRegression             1.0
2     rf                        RandomForestClassifier         1.0
3     gb                        GradientBoostingClassifier     1.0

Top model: lr with test_accuracy = 1.0


In [12]:
# --- Lab 4.2e: Diagnose robustness ---
display, raw = execute_tool("diagnose_robustness", {
    "dataset": "iris",
    "model_name": "rf",
    "noise_levels": [0.01, 0.05, 0.1, 0.2, 0.5],
}, state)
print("DISPLAY:")
print(display)
print("\nRAW noise results:")
for level, metrics in raw["noise_results"].items():
    print(f"  {level}: {metrics}")

DISPLAY:
Robustness diagnosis for 'rf':
  noise_0.01: {'accuracy': 1.0}
  noise_0.05: {'accuracy': 1.0}
  noise_0.1: {'accuracy': 1.0}
  noise_0.2: {'accuracy': 0.9333}
  noise_0.5: {'accuracy': 0.8667}

RAW noise results:
  noise_0.01: {'accuracy': 1.0}
  noise_0.05: {'accuracy': 1.0}
  noise_0.1: {'accuracy': 1.0}
  noise_0.2: {'accuracy': 0.9333}
  noise_0.5: {'accuracy': 0.8667}


In [13]:
# --- Lab 4.2f: Explain importance (permutation) ---
display, raw = execute_tool("explain_importance", {
    "dataset": "iris",
    "model_name": "rf",
    "n_repeats": 10,
}, state)
print("DISPLAY:")
print(display)
print("\nRAW importances:")
for feat, imp in raw["importances"].items():
    bar = "#" * int(imp * 100)
    print(f"  {feat:<25s} {imp:.4f}  {bar}")

DISPLAY:
Permutation importance for 'rf':
  petal length (cm): 0.2500
  petal width (cm): 0.2200
  sepal length (cm): 0.0000
  sepal width (cm): 0.0000

RAW importances:
  petal length (cm)         0.2500  #########################
  petal width (cm)          0.2200  ######################
  sepal length (cm)         0.0000  
  sepal width (cm)          0.0000  


In [14]:
# --- Lab 4.2g: Explain PDP ---
display, raw = execute_tool("explain_pdp", {
    "dataset": "iris",
    "model_name": "rf",
    "features": ["petal length (cm)"],
}, state)
print("DISPLAY:")
print(display)
print("\nRAW PDP keys:", list(raw["pdp"].keys()))

DISPLAY:
Partial dependence for 'rf' on ['petal length (cm)']:
  petal length (cm): grid[0..9]=[1.3, 1.552632, 1.805263, 2.057895, 2.310526, 2.563158, 2.815789, 3.068421, 3.321053, 3.573684]

RAW PDP keys: ['petal length (cm)']


/home/asudjianto/cluster/spark-venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
/home/asudjianto/cluster/spark-venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
/home/asudjianto/cluster/spark-venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
/home/asudjianto/cluster/spark-venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
/home/asudjianto/cluster/spark-venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but RandomForestClassifier 

In [15]:
# --- Lab 4.2h: Explain SHAP ---
display, raw = execute_tool("explain_shap", {
    "dataset": "iris",
    "model_name": "rf",
    "n_samples": 30,
}, state)
print("DISPLAY:")
print(display)
print("\nRAW mean |SHAP|:")
for feat, val in raw["mean_abs_shap"].items():
    print(f"  {feat}: {val:.4f}")

  0%|          | 0/30 [00:00<?, ?it/s]

DISPLAY:
Mean |SHAP| for 'rf' (n=30):
  petal length (cm): 0.2070
  petal width (cm): 0.2008
  sepal length (cm): 0.0207
  sepal width (cm): 0.0083

RAW mean |SHAP|:
  petal length (cm): 0.2070
  petal width (cm): 0.2008
  sepal length (cm): 0.0207
  sepal width (cm): 0.0083


In [16]:
# --- Lab 4.2i: Agent stubs ---
display, raw = execute_tool("recall_memory",
                            {"entity": "iris", "relation": "has_accuracy"}, state)
print(display)
print("RAW:", raw)
print()

display, raw = execute_tool("save_finding", {
    "triples": [
        {"subject": "rf", "predicate": "test_accuracy", "object": "0.9667"},
        {"subject": "rf", "predicate": "task_type", "object": "classification"},
    ]
}, state)
print(display)
print("RAW:", raw)

[STUB] recall_memory(entity='iris', relation='has_accuracy')
Full implementation in Chapter 5.
RAW: {'entity': 'iris', 'relation': 'has_accuracy', 'triples': [], 'stub': True}

[STUB] save_finding: 2 triples received.
Full implementation in Chapter 5.
RAW: {'n_triples': 2, 'triples': [{'subject': 'rf', 'predicate': 'test_accuracy', 'object': '0.9667'}, {'subject': 'rf', 'predicate': 'task_type', 'object': 'classification'}], 'stub': True}


## Exercises

### Exercise 4.1 --- Fairness on Adult Income
Load the Adult Income dataset. Train a GradientBoostingClassifier. Run `diagnose_fairness` with `sex` as the protected attribute. Interpret the AIR.

In [17]:
# --- Exercise 4.1: Fairness on Adult Income ---
state2 = SessionState()

# Load Adult
display, raw = execute_tool("load_dataset",
                            {"name": "adult", "builtin": "adult",
                             "test_size": 0.3, "random_state": 42}, state2)
print(display)

# Train GBM
display, raw = execute_tool("train_model", {
    "dataset": "adult",
    "model_type": "GradientBoostingClassifier",
    "model_name": "gb_adult",
    "params": {"n_estimators": 100, "max_depth": 3, "random_state": 42},
}, state2)
print("\n" + display)

# Diagnose fairness
display, raw = execute_tool("diagnose_fairness", {
    "dataset": "adult",
    "model_name": "gb_adult",
    "protected_attribute": "sex",
}, state2)
print("\n" + display)
print("\n>>> AIR < 0.80 indicates potential disparate impact."
      if not raw.get("air_passes_4_5_rule", True) else
      "\n>>> AIR >= 0.80: passes the 4/5 rule.")

Loaded 'adult': 48842 samples, 14 features, target='income', task=classification
  Train: 34189, Test: 14653



Trained 'gb_adult' (GradientBoostingClassifier):
  train_accuracy: 0.8421
  train_f1_macro: 0.7487
  train_auc: 0.8716
  test_accuracy: 0.8446
  test_f1_macro: 0.7491
  test_auc: 0.8697
  train_test_gap: -0.0025

Fairness diagnosis for 'gb_adult' on 'sex':
  Group positive rates: {'Female': 0.0825, 'Male': 0.1829}
  Group accuracy:       {'Female': 0.9046, 'Male': 0.8149}
  Adverse Impact Ratio: 0.4511
  Demographic parity diff: 0.1004
  AIR >= 0.8 (4/5 rule): False

>>> AIR < 0.80 indicates potential disparate impact.


### Exercise 4.2 --- Predict and Store
Run `predict` for all three Iris models. Verify predictions are stored in `state.predictions`.

In [18]:
# --- Exercise 4.2: Predict for all Iris models ---
for mname in ["lr", "rf", "gb"]:
    display, raw = execute_tool("predict",
                                {"dataset": "iris", "model_name": mname}, state)
    print(f"  {mname}: {raw}")

print(f"\nModels with predictions: {list(state.predictions.keys())}")
print(f"All stored? {set(['lr','rf','gb']).issubset(state.predictions.keys())}")

  lr: {'model_name': 'lr', 'n_predictions': 30, 'unique_values': [0, 1, 2]}
  rf: {'model_name': 'rf', 'n_predictions': 30, 'unique_values': [0, 1, 2]}
  gb: {'model_name': 'gb', 'n_predictions': 30, 'unique_values': [0, 1, 2]}

Models with predictions: ['rf', 'lr', 'gb']
All stored? True


### Exercise 4.3 --- Full Pipeline Validation
Run all 15 tools in sequence on Iris. Verify every raw result is a pure Python dict (JSON-serializable).

In [19]:
# --- Exercise 4.3: Full pipeline, all 15 tools ---
state3 = SessionState()

pipeline = [
    ("load_dataset",        {"name": "iris", "builtin": "iris"}),
    ("data_summary",        {"dataset": "iris"}),
    ("eda_correlation",     {"dataset": "iris", "method": "spearman"}),
    ("preprocess_data",     {"dataset": "iris"}),
    ("feature_select",      {"dataset": "iris", "method": "f_classif", "k": 4}),
    ("train_model",         {"dataset": "iris", "model_type": "RandomForestClassifier",
                             "model_name": "rf_ex3", "params": {"n_estimators": 50, "random_state": 42}}),
    ("predict",             {"dataset": "iris", "model_name": "rf_ex3"}),
    ("leaderboard",         {"dataset": "iris"}),
    ("diagnose_robustness", {"dataset": "iris", "model_name": "rf_ex3",
                             "noise_levels": [0.05, 0.1]}),
    ("diagnose_fairness",   {"dataset": "iris", "model_name": "rf_ex3",
                             "protected_attribute": "sepal length (cm)"}),
    ("explain_importance",  {"dataset": "iris", "model_name": "rf_ex3"}),
    ("explain_pdp",         {"dataset": "iris", "model_name": "rf_ex3",
                             "features": ["petal width (cm)"]}),
    ("explain_shap",        {"dataset": "iris", "model_name": "rf_ex3", "n_samples": 20}),
    ("recall_memory",       {"entity": "iris"}),
    ("save_finding",        {"triples": [{"subject": "rf_ex3", "predicate": "tested", "object": "true"}]}),
]

print(f"Running {len(pipeline)} tools in sequence:")
print("=" * 60)

all_pass = True
for tool_name, args in pipeline:
    display, raw = execute_tool(tool_name, args, state3)
    try:
        json.dumps(raw)
        status = "OK"
    except (TypeError, ValueError) as e:
        status = f"FAIL: {e}"
        all_pass = False
    print(f"  {tool_name:<22s}  {status}  (keys: {list(raw.keys())[:4]}...)")

print(f"\n{'='*60}")
print(f"All 15 tools returned pure Python dicts: {all_pass}")
print("State summary:")
print(f"  datasets:      {list(state3.datasets.keys())}")
print(f"  models:        {list(state3.models.keys())}")
print(f"  results:       {list(state3.results.keys())}")
print(f"  preprocessors: {list(state3.preprocessors.keys())}")
print(f"  predictions:   {list(state3.predictions.keys())}")

Running 15 tools in sequence:
  load_dataset            OK  (keys: ['name', 'n_samples', 'n_features', 'target_name']...)
  data_summary            OK  (keys: ['describe', 'dtypes', 'missing_pct', 'n_rows']...)
  eda_correlation         OK  (keys: ['method', 'correlation']...)
  preprocess_data         OK  (keys: ['impute_method', 'scale_method', 'encode_method', 'n_features_before']...)
  feature_select          OK  (keys: ['method', 'k', 'selected_features', 'importances']...)
  train_model             OK  (keys: ['model_name', 'model_type', 'metrics']...)
  predict                 OK  (keys: ['model_name', 'n_predictions', 'unique_values']...)
  leaderboard             OK  (keys: ['sort_by', 'ranking']...)
  diagnose_robustness     OK  (keys: ['model_name', 'noise_results']...)
  diagnose_fairness       OK  (keys: ['protected_attribute', 'group_positive_rates', 'group_accuracy', 'adverse_impact_ratio']...)


  explain_importance      OK  (keys: ['model_name', 'n_repeats', 'importances']...)
  explain_pdp             OK  (keys: ['model_name', 'features', 'pdp']...)


/home/asudjianto/cluster/spark-venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
/home/asudjianto/cluster/spark-venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
/home/asudjianto/cluster/spark-venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
/home/asudjianto/cluster/spark-venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
/home/asudjianto/cluster/spark-venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but RandomForestClassifier 

  0%|          | 0/20 [00:00<?, ?it/s]

  explain_shap            OK  (keys: ['model_name', 'n_samples', 'mean_abs_shap']...)
  recall_memory           OK  (keys: ['entity', 'relation', 'triples', 'stub']...)
  save_finding            OK  (keys: ['n_triples', 'triples', 'stub']...)

All 15 tools returned pure Python dicts: True
State summary:
  datasets:      ['iris']
  models:        ['rf_ex3']
  results:       ['rf_ex3', 'rf_ex3_robustness', 'rf_ex3_fairness', 'rf_ex3_importance']
  preprocessors: ['iris']
  predictions:   ['rf_ex3']


## Lab 4.3 --- Build the Policy Store

The second half of Chapter 4 trains a *policy* GMS store that encodes which agents may use which tools.  The store is built from `knowlytix/harness/testing/tests/policy_governance_datascience.md` plus a seeded corpus of 241 `(agent, uses_tool, tool)` triples across nine governance relations.

**Architectural note (paper §7).** The policy store uses *relation-conditioned spherical cap admissibility*: each relation learns its own admissible radius $\rho_r$ and its own diagonal conditioning $\Lambda_r$.  A triple $(h, r, t)$ is admitted when $d_{\text{geo}}(c_{h,r}, e_t) \le \rho_r$, where $c_{h,r} = \operatorname{normalize}(\Lambda_r \odot R_r e_h)$.  For functional relations (e.g. `has_risk_tier`) the training shrinkage pulls $\rho_r \to 0$ and the cap reduces to a single point; for set-valued relations (e.g. `uses_tool`, `complies_with`) the radius inflates to absorb the per-head tail cluster.

In [ ]:
# --- Lab 4.3: Build the data-science policy store ---
from knowlytix.harness.testing.policy_store import build_policy_store

policy_store, policy_judge = build_policy_store(
    corpus="data_science",
    stores_dir="gms_agent_builder_ext/lab_stores",
    store_name="ch04_ds_policy",
    ingest_epochs=500,
    device=DEVICE,
    loss_mode="cap",        # paper.tex Section 7
    force_rebuild=True,       # retrain from scratch for Ch04
)

print("\nStore summary:")
for k, v in policy_store.stats().items():
    print(f"  {k}: {v}")


## Lab 4.4 --- Sanity Probe the `uses_tool` Manifold

Chapter 4's authorization story is: reviewers authorised to run their diagnostic tool, modelers authorised to train models, but **no** modeler or reviewer authorised to deploy or approve.  The cap-trained store must (i) admit all four authorised probes with $d_{\text{geo}} \le \rho_{\text{uses\_tool}}$ and (ii) reject all four unauthorised probes with a clear margin.

In [21]:
# --- Lab 4.4: Authorized / unauthorized probes on uses_tool ---
rho_uses = policy_store.cap_radius("uses_tool")
print(f"Learned ρ_uses_tool = {rho_uses:.4f}\n")

authorized_samples = [
    ("weakness_reviewer",    "uses_tool", "diagnose_weakness"),
    ("resilience_reviewer",  "uses_tool", "diagnose_resilience"),
    ("stability_reviewer",   "uses_tool", "diagnose_stability"),
    ("modeler_tree",         "uses_tool", "train_model"),
]
unauthorized_samples = [
    ("modeler_tree",         "uses_tool", "deploy_model"),
    ("eda_analyst",          "uses_tool", "approve_model"),
    ("weakness_reviewer",    "uses_tool", "deploy_model"),
    ("feature_engineer",     "uses_tool", "trigger_retrain"),
]

print("=== Authorized (require d ≤ ρ) ===")
for h, r, t in authorized_samples:
    d = policy_store.score_triple(h, r, t)
    verdict = "OK" if d <= rho_uses else f"FAIL(+{d-rho_uses:.3f})"
    print(f"  {h:22s} {r} {t:22s}  d={d:.4f} [{verdict}]")

print("\n=== Unauthorized (require d > ρ) ===")
for h, r, t in unauthorized_samples:
    d = policy_store.score_triple(h, r, t)
    verdict = "OK" if d > rho_uses else f"FAIL(-{rho_uses-d:.3f})"
    print(f"  {h:22s} {r} {t:22s}  d={d:.4f} [{verdict}]")


Learned ρ_uses_tool = 1.0224

=== Authorized (require d ≤ ρ) ===
  weakness_reviewer      uses_tool diagnose_weakness       d=0.9578 [OK]
  resilience_reviewer    uses_tool diagnose_resilience     d=0.8978 [OK]
  stability_reviewer     uses_tool diagnose_stability      d=0.8894 [OK]
  modeler_tree           uses_tool train_model             d=0.4894 [OK]

=== Unauthorized (require d > ρ) ===
  modeler_tree           uses_tool deploy_model            d=1.8002 [OK]
  eda_analyst            uses_tool approve_model           d=1.7096 [OK]
  weakness_reviewer      uses_tool deploy_model            d=1.6517 [OK]
  feature_engineer       uses_tool trigger_retrain         d=1.7869 [OK]


## Lab 4.5 --- The $\rho_r$ Topology Diagnostic

One practical virtue of relation-conditioned caps is that the learned radii themselves are an audit artefact.  A functional relation (avg tails per head $=1$) should settle at small $\rho_r$; a set-valued relation (avg tails $>1$) should inflate $\rho_r$ to cover its tail cluster.  Inspecting the sorted radius table verifies that the trained store has discovered the corpus topology without any external labelling.

In [22]:
# --- Lab 4.5: Per-relation cap radius table ---
radii = policy_store.cap_radii_all()
print(f"{'relation':30s} {'ρ_r':>8s}")
print("-" * 42)
for name, rho in sorted(radii.items(), key=lambda x: -x[1]):
    print(f"{name:30s} {rho:>8.4f}")


relation                            ρ_r
------------------------------------------
complies_with                    1.0735
uses_tool                        1.0224
requires_check                   1.0084
reads_data                       0.9942
can_approve                      0.8264
violates                         0.8143
in_section                       0.7730
has_risk_tier                    0.7378
has_seniority                    0.6993
has_authority_level              0.6858
has_function                     0.6637
has_role                         0.6582
has_side_effects                 0.6578
has_purpose                      0.6408
escalates_to                     0.6398
has_description                  0.6385
restricted_to_tier               0.6099
has_concern                      0.6010
has_pass_criterion_(illustrative)   0.5885
writes_data                      0.5818
fails                            0.5652
passes                           0.5265
is_weak                          0

## Exercise 4.4 --- Probe a Different Relation

The sanity probe above tested `uses_tool`.  Repeat it on `complies_with`: each exemplar model has its own authorised set of compliance standards, and the relation's learned radius $\rho_{\text{complies\_with}}$ must separate them from cross-model corruptions.  No re-calibration is needed: the cap store carries one radius per relation.

In [23]:
# --- Exercise 4.4: probe complies_with ---
rho_cw = policy_store.cap_radius("complies_with")
print(f"rho_complies_with = {rho_cw:.4f}\n")

authorized = [
    ("high_risk_classification", "complies_with", "weakness_identification_standard"),
    ("high_risk_classification", "complies_with", "fairness_standard"),
    ("regulated_lending_model",  "complies_with", "explainability_standard"),
    ("clinical_triage_model",    "complies_with", "privacy_standard"),
]
unauthorized = [
    ("high_risk_classification",  "complies_with", "privacy_standard"),
    ("conservative_credit_model", "complies_with", "fairness_standard"),
    ("monitored_churn_model",     "complies_with", "stability_standard"),
    ("validated_baseline_model",  "complies_with", "weakness_identification_standard"),
]

print("=== Authorized (require d ≤ ρ) ===")
for h, r, t in authorized:
    d = policy_store.score_triple(h, r, t)
    verdict = "OK" if d <= rho_cw else f"FAIL(+{d-rho_cw:.3f})"
    print(f"  {h:27s} -> {t:38s} d={d:.4f}  [{verdict}]")

print("\n=== Unauthorized (require d > ρ) ===")
for h, r, t in unauthorized:
    d = policy_store.score_triple(h, r, t)
    verdict = "OK" if d > rho_cw else f"FAIL(-{rho_cw-d:.3f})"
    print(f"  {h:27s} -> {t:38s} d={d:.4f}  [{verdict}]")


rho_complies_with = 1.0735

=== Authorized (require d ≤ ρ) ===
  high_risk_classification    -> weakness_identification_standard       d=0.5509  [OK]
  high_risk_classification    -> fairness_standard                      d=0.6470  [OK]
  regulated_lending_model     -> explainability_standard                d=0.4748  [OK]
  clinical_triage_model       -> privacy_standard                       d=0.8387  [OK]

=== Unauthorized (require d > ρ) ===
  high_risk_classification    -> privacy_standard                       d=1.5462  [OK]
  conservative_credit_model   -> fairness_standard                      d=1.2076  [OK]
  monitored_churn_model       -> stability_standard                     d=1.4038  [OK]
  validated_baseline_model    -> weakness_identification_standard       d=1.4788  [OK]


## Exercise 4.5 --- Authorized Tails Rank Before Unauthorized

Lab 4.4 picked four authorised and four unauthorised probe triples.  This exercise checks the admissibility property *at scale*: for a given head, score every entity as a candidate tail, rank ascending, and verify that the seeded authorised tails occupy the top of the list.

In [24]:
# --- Exercise 4.5: ranking check for an authorised head ---
import torch

head = "weakness_reviewer"
relation = "uses_tool"

h_idx = policy_store.adapter.entity_to_idx[head]
r_idx = policy_store.adapter.relation_to_idx[relation]

device = next(policy_store.model.parameters()).device
with torch.no_grad():
    scores = policy_store.model.score_all_tails_cap(
        torch.tensor([h_idx], device=device),
        torch.tensor([r_idx], device=device),
    ).squeeze(0).cpu()

ranked = sorted(
    [(policy_store.adapter.idx_to_entity[i], s.item())
     for i, s in enumerate(scores)],
    key=lambda kv: kv[1],
)

authorised = {
    t for (h, r, t)
    in zip(policy_store.adapter.heads.tolist(),
           policy_store.adapter.relations.tolist(),
           policy_store.adapter.tails.tolist())
    if h == h_idx and r == r_idx
}
authorised_names = {policy_store.adapter.idx_to_entity[t] for t in authorised}

print(f"Top-10 candidates for ({head}, {relation}, ?):")
for rank, (name, s) in enumerate(ranked[:10], start=1):
    flag = "AUTHORISED" if name in authorised_names else ""
    print(f"  {rank:2d}. {name:28s} score={s:.4f}  {flag}")


Top-10 candidates for (weakness_reviewer, uses_tool, ?):
   1. explain_pfi                  score=0.5820  AUTHORISED
   2. train_model                  score=0.6512  AUTHORISED
   3. predict                      score=0.7285  AUTHORISED
   4. diagnose_weakness            score=0.9578  AUTHORISED
   5. leaderboard                  score=1.0711  
   6. explain_shap                 score=1.0990  
   7. diagnose_stability           score=1.1843  
   8. diagnose_accuracy            score=1.2249  
   9. diagnose_resilience          score=1.2647  
  10. trigger_retrain              score=1.3498  
